In [1]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

In [2]:
import time
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dropout, Dense
from tabulate import tabulate
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
import glob

In [3]:
folder_path = './Networks'

files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

datasets = [pd.read_csv(os.path.join(folder_path, file)) for file in files]

print(f"Loaded {len(datasets)} datasets")
print("\nSample from the first dataset:")
datasets[0].head()

C:\Users\pc\AppData\Local\Temp\ipykernel_14812\3772580079.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  datasets = [pd.read_csv(os.path.join(folder_path, file)) for file in files]


Loaded 7 datasets

Sample from the first dataset:


,ts,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,1554198358,3.122.49.24,1883,192.168.1.152,52976,tcp,-,80549.530260,1762852,41933215,...,0,0,-,-,-,bad_TCP_checksum,-,F,0,normal
1,1554198358,192.168.1.79,47260,192.168.1.255,15600,udp,-,0.000000,0,0,...,0,0,-,-,-,-,-,-,0,normal
2,1554198359,192.168.1.152,1880,192.168.1.152,51782,tcp,-,0.000000,0,0,...,0,0,-,-,-,bad_TCP_checksum,-,F,0,normal
3,1554198359,192.168.1.152,34296,192.168.1.152,10502,tcp,-,0.000000,0,0,...,0,0,-,-,-,-,-,-,0,normal
4,1554198362,192.168.1.152,46608,192.168.1.190,53,udp,dns,0.000549,0,298,...,0,0,-,-,-,bad_UDP_checksum,-,F,0,normal


In [4]:
def preprocess_data(df):
    df = df.drop(['src_ip', 'dst_ip', 'dns_query'], axis=1, errors='ignore')
    
    df.fillna(0, inplace=True)
    
    label_encoders = {}
    for column in df.select_dtypes(include=['object']).columns:
        df[column] = df[column].astype(str)
        
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le

    return df

datasets = [preprocess_data(dataset) for dataset in datasets]


In [5]:
df = pd.concat(datasets, axis=0)
X = df.drop('label', axis=1, errors='ignore')  
y = df['label'] 

In [6]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [18]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

def hyperparameter_tuning(model, param_grid, X_train, y_train, search_type="grid", n_iter=100, cv=3, scoring='accuracy'):
    if search_type == "grid":
        search = GridSearchCV(estimator=model, param_grid=param_grid, cv=cv, n_jobs=-1, verbose=2, scoring=scoring)
    elif search_type == "random":
        search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, n_iter=n_iter, 
                                    cv=cv, n_jobs=-1, verbose=2, scoring=scoring, random_state=42)
    
    search.fit(X_train, y_train)
    
    return search.best_estimator_, search.best_params_


def train_tunned_model(model, X_train, y_train, X_test, param_grid=None, tune=False, search_type="grid"):
 
    start_time = time.time()  
    if tune and param_grid is not None:
        model, best_params = hyperparameter_tuning(model, param_grid, X_train, y_train, search_type)

    model.fit(X_train, y_train) 

    training_time = time.time() - start_time  

    y_pred = model.predict(X_test)  

    return y_pred, training_time

In [8]:
def calculate_metrics(y_test, y_pred, training_time, average='weighted', model_name='Random Forest'):
    accuracy = accuracy_score(y_test, y_pred)
    error_rate = 1 - accuracy
    precision = precision_score(y_test, y_pred, average=average)
    recall = recall_score(y_test, y_pred, average=average)
    f1 = f1_score(y_test, y_pred, average=average)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel() if len(confusion_matrix(y_test, y_pred).ravel()) == 4 else (None, None, None, None)
    
    false_positive_rate = fp / (fp + tn) if fp is not None else "N/A"
    specificity = tn / (tn + fp) if tn is not None else "N/A"

    headers = ["Model", "Training Time (s)", "Accuracy (%)", "Error Rate (%)", 
               "Precision", "Recall", "F1-Score", "False Positive Rate", "Specificity"]
    data = [[
        model_name, 
        f"{training_time:.2f}", 
        f"{accuracy * 100:.2f}", 
        f"{error_rate * 100:.2f}", 
        f"{precision:.2f}", 
        f"{recall:.2f}", 
        f"{f1:.2f}", 
        f"{false_positive_rate:.2f}" if false_positive_rate != "N/A" else "0", 
        f"{specificity:.2f}" if specificity != "N/A" else "0"
    ]]

    return tabulate(data, headers=headers, tablefmt="grid")


In [9]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |              560.46 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [19]:
rf_param_grid = {
    'n_estimators': [5, 10],       # Reduce number of trees significantly
    'max_depth': [2, 3],            # Limit tree depth
    'min_samples_split': [20, 50],  # Require more samples before splitting
    'min_samples_leaf': [10, 20]     # Minimum samples in leaf nodes
}

y_pred, training_time = train_tunned_model(
    RandomForestClassifier(random_state=42),
    X_train, y_train, X_test,
    param_grid=rf_param_grid, tune=True, search_type="grid"
)

metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

Fitting 3 folds for each of 16 candidates, totalling 48 fits


In [9]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |              550.01 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [ ]:
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10],  # Smaller C means stronger regularization (less overfitting)
    'penalty': ['l2']         # L2 regularization (Ridge)
}

y_pred, training_time = train_model(
    LogisticRegression(random_state=42, max_iter=1000),
    X_train, y_train, X_test,
    param_grid=lr_param_grid, tune=True, search_type="grid"
)

metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

In [16]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |              525.55 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [212]:
svm_param_grid = {
    'C': [0.1, 1, 10],        # Lower C will increase regularization and prevent overfitting
    'gamma': [0.001, 0.01],   # Reduce gamma to smooth decision boundaries
    'kernel': ['rbf']    
}

y_pred, training_time = train_tunned_model(
    SVC(random_state=42),
    X_train, y_train, X_test,
    param_grid=svm_param_grid, tune=True, search_type="grid"
)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


In [11]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |              550.57 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [ ]:
sgd_param_grid = {
    'alpha': [0.0001, 0.001, 0.01],  # Regularization strength
    'penalty': ['l2', 'l1'],         # L2 is default; L1 can increase sparsity
    'learning_rate': ['optimal', 'invscaling']
}

y_pred, training_time = train_tunned_model(
    SGDClassifier(random_state=42),
    X_train, y_train, X_test,
    param_grid=sgd_param_grid, tune=True, search_type="random"
)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

In [12]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |               542.4 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [ ]:
ada_param_grid = {
    'n_estimators': [50, 100, 200],  # Number of weak learners (lower complexity with fewer learners)
    'learning_rate': [0.01, 0.1, 1]  # Control the contribution of each learner
}

y_pred, training_time = train_model(
    AdaBoostClassifier(random_state=42),
    X_train, y_train, X_test,
    param_grid=ada_param_grid, tune=True, search_type="grid"
)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

In [15]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |               542.4 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [ ]:
gb_param_grid = {
    'n_estimators': [50, 100, 150],     # Reduce number of trees
    'learning_rate': [0.01, 0.1],       # Lower learning rate to prevent overfitting
    'max_depth': [3, 5],                # Control depth of trees
    'min_samples_split': [5, 10],       # Prevent overfitting by requiring more samples
    'min_samples_leaf': [2, 4]          # Minimum samples in leaf nodes
}

y_pred, training_time = train_model(
    GradientBoostingClassifier(random_state=42),
    X_train, y_train, X_test,
    param_grid=gb_param_grid, tune=True, search_type="grid"
)

metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

In [14]:
y_train_str = y_train.astype(str)
y_test_str = y_test.astype(str)

combined_labels = np.concatenate((y_train_str, y_test_str))

label_encoder = LabelEncoder()
label_encoder.fit(combined_labels)

y_train_encoded = label_encoder.transform(y_train_str)
y_test_encoded = label_encoder.transform(y_test_str)

    # label_encoder = LabelEncoder()
    # y_train_encoded = label_encoder.fit_transform(y_train.astype(str))  
    # y_test_encoded = label_encoder.transform(y_test.astype(str))  

In [142]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Accuracy: 85.15%


In [143]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               18.19 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [144]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 753us/step
Accuracy: 85.15%


In [145]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Depp Artificial Neural Network')
print(metrics_table)

+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Depp Artificial Neural Network |               47.62 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [146]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))
 
    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=64, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

X_train_cnn_lstm = X_train.values.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.values.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
Accuracy: 85.15%


In [147]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |              103.91 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## NORMALIZATION OF X INPUT

In [148]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [149]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)

In [150]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                1.65 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [151]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)

In [152]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                1.64 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [153]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)

In [154]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                1.61 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [155]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)

In [156]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                1.74 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [157]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)

In [158]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                1.55 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [159]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)

In [160]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                1.56 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [161]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 0s 636us/step
Accuracy: 85.15%


In [162]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               14.98 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [163]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 730us/step
Accuracy: 85.15%


In [164]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               38.73 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [165]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
Accuracy: 85.15%


In [166]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |               70.68 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## USING BEST FEATURES

In [167]:
from sklearn.feature_selection import f_classif, SelectKBest

imputer = SimpleImputer(strategy='mean')

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

fs = SelectKBest(f_classif, k=28)
fs.fit(X_train_imputed, y_train)

X_train = fs.transform(X_train_imputed)
X_test = fs.transform(X_test_imputed)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: UserWarning: Features [25 26 28 29 30 32 35 36 39 41] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw
c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [168]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                1.34 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [169]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                1.32 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [170]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                1.31 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [171]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                1.35 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [172]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                1.32 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [173]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)

In [174]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                 1.3 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [175]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 732us/step
Accuracy: 100.00%


In [176]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               14.74 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [177]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 737us/step
Accuracy: 100.00%


In [178]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               38.48 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [179]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
Accuracy: 100.00%


In [180]:
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |               50.44 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


## USING PCA FOR THE MODELS

In [181]:
from sklearn.decomposition import PCA

In [182]:
pca = PCA(n_components=12)


imputer = SimpleImputer(strategy='mean')

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)
x_train = pca.fit_transform(X_train)
x_test = pca.transform(X_test)

In [183]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                1.38 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [184]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                1.42 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [185]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                1.44 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [186]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                1.41 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [187]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                1.37 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [188]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                1.35 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [189]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 0s 665us/step
Accuracy: 100.00%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               15.36 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [190]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 797us/step
Accuracy: 100.00%
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               39.04 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [191]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Accuracy: 100.00%
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |               52.49 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


## USING SMOTE FOR THE MODELS 

In [192]:
from imblearn.over_sampling import SMOTE

In [193]:
X_train = X_train[:min(len(X_train), len(y_train))]
y_train = y_train[:min(len(X_train), len(y_train))]

In [194]:
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [195]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)

metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                2.42 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [196]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                2.33 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [197]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                2.33 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [198]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                2.37 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [199]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                2.32 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [200]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                2.33 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [201]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 0s 655us/step
Accuracy: 100.00%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               25.09 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [202]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 768us/step
Accuracy: 100.00%
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               63.66 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [203]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Accuracy: 100.00%
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |               91.04 |            100 |                0 |           1 |        1 |          1 |                     0 |             1 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
